# `signals` — Raw Alpha Factors

We compare two classic cross-sectional anomalies that pull in **opposite** directions at short horizons — the perfect pair for showing how transaction costs decide which signal is actually tradeable.

### 1. Momentum (12–1)
The 11-month return ending one month ago:
$$ \text{MOM}_{i,t} = \frac{P_{i,t-21}}{P_{i,t-252}} - 1 $$
We **skip the most recent month** (the `-21` lag) precisely because of the short-term reversion effect below — including it would contaminate a slow signal with a fast, opposite one. Momentum is a *slow* signal: rankings drift gradually, so turnover (and therefore cost) is low.

### 2. Short-term reversion (5-day)
The negative of the trailing 5-day return:
$$ \text{REV}_{i,t} = -\sum_{k=0}^{4} r_{i,t-k} $$
Bet: names that just popped tend to give some back. This is a *fast* signal — the ranking flips every few days, so it has **high turnover** and will be punished by costs.

*(Definitions only — safe to `%run` from `main.ipynb`.)*

In [ ]:
"""Raw cross-sectional alpha factors: 12-1 momentum and 5-day reversion."""
import pandas as pd


def generate_momentum_12_1(prices: pd.DataFrame) -> pd.DataFrame:
    """12-1 momentum: the 11-month return that ends about one month ago."""
    # 252 trading days ~ 12 months, 21 ~ 1 month. Comparing the price 21 days ago with the price
    # 252 days ago gives the return over that 11-month stretch.
    #
    # Why deliberately skip the most recent month instead of just using the last 12 months?
    # Because the last month is dominated by SHORT-TERM REVERSION -- the very effect we test as
    # the second signal, and it points the opposite way. Including it would mix a slow signal
    # with a fast, opposing one and blur both. This "skip a month" convention is why the factor
    # is called 12-1 rather than simply 12-month momentum.
    return prices.shift(21) / prices.shift(252) - 1.0


def generate_short_term_reversion(returns: pd.DataFrame) -> pd.DataFrame:
    """Short-term mean reversion: minus the trailing 5-day return."""
    # Add up the last 5 daily returns, then flip the sign. After flipping, a stock that FELL
    # over the past week scores HIGH (we are betting it bounces back) and a stock that rallied
    # scores low. The sign flip is what turns "past return" into a reversion signal, so that a
    # high score always means "expected to outperform" -- the same convention as momentum, which
    # keeps the sign of the IC interpretable across both factors.
    #
    # Summing simple returns is a slight approximation to the true compounded 5-day return, but
    # everything downstream ranks stocks, and the sum is monotone in the compounded return --
    # same ordering, so the rank IC is identical.
    cum_5d = returns.rolling(window=5).sum()
    return -cum_5d


print("signal helpers ready: generate_momentum_12_1(), generate_short_term_reversion()")